# 🚗 Audi A1 Price Prediction — İkinci El Araç Fiyat Tahmini

## 🎯 Problem Tanımı
Bu çalışmada Audi A1 ikinci el araç ilanlarındaki teknik ve ilan özelliklerini (model yılı, kilometre, motor hacmi, beygir gücü, vites tipi, yakıt türü ve sahip sayısı) kullanarak **araç satış fiyatını (`Price(£)`) tahmin eden** bir doğrusal regresyon (`LinearRegression`) modeli geliştiriyoruz.

## 📊 Veri Seti
- **Kaynak:** [Kaggle — Audi A1 Listings](https://www.kaggle.com/datasets/aishwarya2210/audi-a1-listings) (`Audi_A1_listings.csv`)
- **Boyut:** 471 satır, 16 sütun
- **Hedef Değişken:** `Price(£)` (Araç satış fiyatı, Sterlin cinsinden)

## 💡 Bu Notebook'ta Öğrenilecekler
- **Veri sızıntısı (Data Leakage) önleme:** Hedef değişkenden türetilmiş sütunları (`PriceRank`, `PPY`, `Score`, `PPYRank`) tespit edip model dışında bırakma
- **Metin ve birim temizleme:** `"1.6L"` gibi birim soneklerini (`"L"`) temizleyerek sayısal veri tipine dönüştürme
- **One-Hot Encoding:** Kategorik değişkenlerin `pd.get_dummies` ile modellenebilir hale getirilmesi
- **Tam regresyon değerlendirmesi:** `R²` skoruna ek olarak `RMSE` ve `MAE` metriklerinin hesaplanması

## 1. Ortam Kontrolü ve Kütüphanelerin Yüklenmesi
Veri seti yolu kontrol edilir ve regresyon modellemesi için gerekli kütüphaneler içe aktarılır.

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/jacklacey/audi-a1-listings/Audi_A1_listings.csv


In [2]:
import pandas as pd 
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


## 2. Veri Setinin Yüklenmesi ve İlk İnceleme
Veri seti okunur ve ilk 3 satır üzerinden sütun yapısı incelenir.

In [3]:
df = pd.read_csv((__import__("glob").glob("/kaggle/input/**/Audi_A1_listings.csv", recursive=True)[0] if not os.path.exists("/kaggle/input/audi-a1-listings/Audi_A1_listings.csv") else "/kaggle/input/audi-a1-listings/Audi_A1_listings.csv"))

In [4]:
df.head(3)

,index,Year,Type,Mileage(miles),Engine,PS,Transmission,Fuel,Number_of_Owners,Price(£),href,PPY,MileageRank,PriceRank,PPYRank,Score
0,0,2018.0,Hatchback,44000.0,1.6L,114.398422,Manual,Diesel,1,14995.0,https://www.autotrader.co.uk/car-details/20221...,2499.166667,215,163,340,718
1,4,2016.0,Hatchback,42596.0,1.0L,93.688363,Manual,Petrol,3,10755.0,https://www.autotrader.co.uk/car-details/20221...,2688.750000,222,330,276,828
2,7,2015.0,Hatchback,42700.0,1.4L,123.274162,Manual,Petrol,2,10799.0,https://www.autotrader.co.uk/car-details/20221...,3599.666667,221,327,94,642


## 3. Veri Sızıntısı (Data Leakage) Önleme ve Sütun Seçimi
Veri setindeki bazı sütunlar **doğrudan hedef değişken olan `Price(£)` bilgisinden türetilmiştir** veya modelleme açısından anlamsız kimlik/sıralama verisi taşır. Bu sütunların çıkarılma nedenleri şunlardır:

- **`PriceRank`:** Araçların fiyatına göre veri seti içindeki sıralamasını verir. Tahmin edilmek istenen fiyatı doğrudan sızdırdığı için (**data leakage**) çıkarılmıştır.
- **`PPY` (Price Per Year) ve `PPYRank`:** Yıllık maliyet (`Price / yaş`) ve bunun sıralaması doğrudan satış fiyatı (`Price(£)`) kullanılarak hesaplandığından test sırasında gerçek hayatta bilinemez; veri sızıntısına yol açar.
- **`Score`:** İlan puanlama skoru `MileageRank`, `PriceRank` ve `PPYRank` sıralamalarının birleşiminden türetildiği için hedef fiyat bilgisini dolaylı olarak modele taşır.
- **`MileageRank`:** Veri seti içi görece sıralama bilgisi taşır ve yeni/tekil ilanlarda genellenebilir değildir.
- **`index` ve `href`:** Sıra numarası ve ilan URL'si olup fiyat tahmininde belirleyici bir özellik değildir.

> Bu sütunlar çıkarıldığında modelin skoru düşebilir, ancak elde edilen performans gerçekçi ve güvenilirdir.

In [5]:
df=df.drop(columns=["index","href","MileageRank","PriceRank","PPYRank","Score","PPY"]);

## 4. Eksik Veri Kontrolü ve Veri Tipi Temizliği
Eksik değer bulunup bulunmadığı ve sütun veri tipleri kontrol edilir.

In [6]:
df.isnull().sum()

Year                0
Type                0
Mileage(miles)      0
Engine              0
PS                  0
Transmission        0
Fuel                0
Number_of_Owners    0
Price(£)            0
dtype: int64

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 471 entries, 0 to 470
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Year              471 non-null    float64
 1   Type              471 non-null    object 
 2   Mileage(miles)    471 non-null    float64
 3   Engine            471 non-null    object 
 4   PS                471 non-null    float64
 5   Transmission      471 non-null    object 
 6   Fuel              471 non-null    object 
 7   Number_of_Owners  471 non-null    int64  
 8   Price(£)          471 non-null    float64
dtypes: float64(4), int64(1), object(4)
memory usage: 33.2+ KB


`Engine` sütunundaki değerler `"1.6L"`, `"1.0L"` şeklinde metin (`object`) formatındadır. `"L"` birimi kaldırılıp `pd.to_numeric()` ile sayısal (`float`) tipe dönüştürülür.

In [8]:
df["Engine"]=df["Engine"].str.replace("L","") 

In [9]:
df["Engine"]=pd.to_numeric(df["Engine"]) 

In [10]:
df.head(3)

,Year,Type,Mileage(miles),Engine,PS,Transmission,Fuel,Number_of_Owners,Price(£)
0,2018.0,Hatchback,44000.0,1.6,114.398422,Manual,Diesel,1,14995.0
1,2016.0,Hatchback,42596.0,1.0,93.688363,Manual,Petrol,3,10755.0
2,2015.0,Hatchback,42700.0,1.4,123.274162,Manual,Petrol,2,10799.0


## 5. Kategorik Değişkenlerin Kodlanması ve Sütun İsimlendirme
`Type`, `Transmission` ve `Fuel` sütunları `pd.get_dummies(..., drop_first=True)` ile dummy değişkenlere çevrilir ve sütun adları okunabilir Türkçe isimlerle güncellenir.

In [11]:
df=pd.get_dummies(df,columns=["Type","Transmission","Fuel"],drop_first=True)

In [12]:
df.head(3)

,Year,Mileage(miles),Engine,PS,Number_of_Owners,Price(£),Transmission_Manual,Fuel_Petrol
0,2018.0,44000.0,1.6,114.398422,1,14995.0,True,False
1,2016.0,42596.0,1.0,93.688363,3,10755.0,True,True
2,2015.0,42700.0,1.4,123.274162,2,10799.0,True,True


In [13]:
df.columns=["Yıl","Mil","Motor","Ps","Sahip","Fiyat","Vites_Manuel","Depo"]

In [14]:
df.head()

,Yıl,Mil,Motor,Ps,Sahip,Fiyat,Vites_Manuel,Depo
0,2018.0,44000.0,1.6,114.398422,1,14995.0,True,False
1,2016.0,42596.0,1.0,93.688363,3,10755.0,True,True
2,2015.0,42700.0,1.4,123.274162,2,10799.0,True,True
3,2014.0,86000.0,1.6,103.550296,3,7490.0,True,False
4,2014.0,104310.0,1.6,103.550296,3,7400.0,True,False


## 6. Eğitim / Test Ayrımı ve Model Eğitimi
Bağımsız değişkenler (`x`) ve hedef değişken (`y = Fiyat`) ayrılır. Veri %80 eğitim, %20 test olacak şekilde bölünerek `LinearRegression` modeli eğitilir.

In [15]:
y= df["Fiyat"]
x= df.drop("Fiyat",axis=1)


In [16]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
lm=LinearRegression()
model=lm.fit(x_train,y_train)
model=lm.score(x_test,y_test)

# Daha düşük skor çıktı fakat daha güvenilir.

## 7. Model Değerlendirme Metrikleri (R², RMSE, MAE)
Yalnızca `R²` skoruna bakmak yerine, tahmin hatalarının para birimi (`£`) cinsinden büyüklüğünü görmek için **RMSE** ve **MAE** metrikleri de hesaplanır.

In [17]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
y_pred = lm.predict(x_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = lm.score(x_test, y_test)
print(f"R²:   {r2:.4f}")
print(f"RMSE: {rmse:,.2f}")
print(f"MAE:  {mae:,.2f}")

R²:   0.9016
RMSE: 1,366.36
MAE:  1,140.62


*(Not: Aşağıdaki yorum satırına alınmış hücrede tüm veri üzerinde bölme yapmadan eğitim denenmiştir; ancak eğitim verisi üzerinde değerlendirme aşırı öğrenmeyi gizleyeceği için yukarıdaki `train_test_split` yaklaşımı tercih edilmiştir.)*

In [18]:
#lm=LinearRegression()
#model=lm.fit(x,y)
#model.score(x,y)

## 📝 Özet ve Değerlendirme

### Neler Öğrenildi?
- **Veri Sızıntısını (Data Leakage) Önleme:** `PriceRank`, `PPY`, `PPYRank` ve `Score` gibi hedef değişkenden türetilmiş sütunların çıkarılması, modelin gerçek dünyada kullanılamayacak yapay bir yüksek skor üretmesini engelledi.
- **Metin Temizleme:** `"1.6L"` formatındaki motor hacmi bilgisinden birim karakteri (`"L"`) temizlenerek modelin sayısal büyüklük ilişkisini öğrenebilmesi sağlandı.
- **Doğru Değerlendirme:** `train_test_split` ile ayrılmış test seti üzerinde `R²`, `RMSE` ve `MAE` metrikleri birlikte raporlandı.

### Regresyon Metriklerinin Anlamı
- **R² (Belirlilik Katsayısı):** Modelin araç fiyatlarındaki toplam değişkenliğin ne kadarını açıklayabildiğini gösterir (0 ile 1 arasında, 1'e yakın olması daha iyidir).
- **RMSE (Kök Ortalama Kare Hata):** Tahmin hatalarının karesinin ortalamasının kareköküdür (`£` cinsinden). Büyük hatalara daha yüksek ağırlık verir.
- **MAE (Ortalama Mutlak Hata):** Modelin tahminlerinin gerçek fiyattan ortalama kaç Sterlin (`£`) saptığını doğrudan gösterir.